# 01 — Dataset Preparation for German→English Podcast Translation

**What this notebook does and why it exists**

Before we can fine-tune anything, we need data that matches our task — conversational spoken German translated into natural English. This notebook downloads, filters, deduplicates, and blends several publicly available parallel corpora, then splits the result into train, validation, and test sets. A sacred held-out test set (drawn from spoken-language sources) is written to disk and never touched during training.

We save everything to Google Drive so the subsequent training notebooks can pick up exactly where this one leaves off.

**Runtime note:** This notebook targets a Colab TPU runtime, but the data-preparation cells themselves run on CPU. You do not need to change the runtime type if you just want to prepare data — TPU is selected so that you can run all notebooks in the same Colab session without switching.

---
## Research Notes: Choosing the Base Model

We evaluated four candidate models in the 1–2 B parameter range:

| Model | Params | Training tokens | Multilingual | Licence | MLX support |
|---|---|---|---|---|---|
| **Qwen2.5-1.5B-Instruct** | 1.5B | 18T | ✅ Strong (29+ languages) | Apache 2.0 | ✅ Excellent |
| SmolLM2-1.7B-Instruct | 1.7B | 11T | ❌ Limited | Apache 2.0 | ✅ Good |
| Gemma-3-1B-IT | 1B | 2T | ✅ Strong (140 languages) | Gemma licence | ✅ Good |
| Qwen3-1.7B | 1.7B | ~36T | ✅ Strong | Apache 2.0 | ✅ Good |

**Winner: `Qwen/Qwen2.5-1.5B-Instruct`**

**Reasons:**
1. **Translation capability.** Qwen2.5 was trained on 18 trillion tokens with explicit multilingual coverage and instruction-following alignment. The Qwen2.5 technical report documents strong multilingual FLORES+ scores. Among sub-2B models, Qwen2.5 shows the best out-of-the-box German-to-English translation when prompted appropriately.
2. **Licence.** Apache 2.0 — you can redistribute fine-tuned checkpoints commercially without restriction. Gemma's licence prohibits use in certain competitive AI services, which is a non-trivial constraint for redistribution.
3. **Ecosystem.** Unsloth explicitly supports Qwen2.5 for both SFT and GRPO training. The `mlx-community` has multiple pre-converted Qwen2.5 4-bit models demonstrating strong MLX compatibility. No known architecture issues with `mlx-lm`.
4. **Context window.** 128K tokens — important for long podcast segments.

**What we gave up:** Gemma-3-1B is marginally better at the quality-efficiency trade-off for *error detection* tasks (arXiv 2511.09748), but this advantage does not clearly transfer to *generation* tasks. SmolLM2 excels at benchmarks designed for English-centric tasks and is explicitly not recommended for production translation by HuggingFace.

---
## Research Notes: Dataset Selection

Our goal is a model that produces *natural, conversational English* from spoken German. Register is the most important consideration. We examined the following corpora:

| Dataset | Size (DE→EN) | Licence | Register | Notes |
|---|---|---|---|---|
| **MuST-C v3** | ~496 hrs / ~250K segments | CC BY-NC-ND 4.0 | Conversational (TED talks) | Best proxy for podcast speech. Careful, informal delivery. |
| **CoVoST-2** | ~180K segments | CC0 | Conversational (crowdsourced) | CC0 = fully open. Diverse accents, natural phrasing. |
| **OPUS-Books** | ~3M sentence pairs | Various open | Mixed/literary | Clean human translations; used as informal vocabulary top-up. Apply filters. |
| WMT News (2014–2023) | Tens of millions | Various open | Formal journalistic | **Poor register match** — journalistic prose is too far from spoken language. Use sparingly. |
| Europarl v10 | ~1.9M | CC BY 4.0 | Formal parliamentary | **Poor register match** — stiff, formulaic speech. Actively harmful for our task. |
| OPUS-100 | ~1M (DE-EN) | Various | Mixed | Useful as a top-up but register is inconsistent. |

**Rejected datasets:** Europarl is the canonical example of register mismatch. A model trained heavily on Europarl learns to produce sentences like *"I would like to call upon the Commission to examine whether..."* — entirely unlike spoken German.

### Recommended blend

| Source | Proportion | Rationale |
|---|---|---|
| MuST-C v3 DE→EN | **40%** | Best register match; human-verified translations of spoken content |
| CoVoST-2 DE→EN | **30%** | CC0 licence (ideal), diverse speakers, genuinely conversational |
| OPUS-Books (filtered) | **30%** | Scale; fills in vocabulary gaps not covered by MuST-C and CoVoST-2 |

**Why no WMT news data?** We deliberately exclude it. The risk of register contamination outweighs the benefit of extra parallel sentences. A model that learns that German → stiff journalistic English is a valid output distribution will transfer that style to podcasts.

**Sacred test set:** We hold out 500 segments from MuST-C v3 and CoVoST-2 combined. These are *never* seen during training or validation. They serve as our ground-truth measurement of real-world quality at the end of the project.

In [ ]:
# ── Mount Google Drive ───────────────────────────────────────────────────────
# All outputs are written here so they survive Colab session restarts.
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/podcast_translation'
DATA_DIR = os.path.join(BASE_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Data directory: {DATA_DIR}')

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# Pinned versions ensure reproducibility. If you need to update, note the new
# version here and check for breaking changes in the datasets / tokenizers APIs.
!pip install -q \
    datasets==2.21.0 \
    transformers==4.45.0 \
    sacremoses==0.1.1 \
    pandas==2.2.2 \
    matplotlib==3.9.2 \
    tqdm==4.66.4 \

print('Dependencies installed.')

In [ ]:
import json
import random
import hashlib
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from transformers import AutoTokenizer

random.seed(42)
TOKENIZER_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
print('Imports complete.')

In [ ]:
# ── Load tokeniser ───────────────────────────────────────────────────────────
# We use the same tokeniser as the model we will train. Token-length filtering
# using the actual tokeniser is more accurate than word-count heuristics,
# because Qwen2.5 uses byte-pair encoding which splits German compound words
# differently from naive whitespace splitting.
tokeniser = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
print(f'Vocabulary size: {tokeniser.vocab_size:,}')

---
## Decision note: Why filter to 5–120 German tokens?

- **Lower bound (5 tokens):** Segments shorter than five tokens are almost certainly sentence fragments, headers, or artefacts (e.g., *(Applause)*, single words). Including them trains the model to produce trivial outputs and inflates validation metrics.
- **Upper bound (120 tokens):** Typical podcast sentences rarely exceed 120 tokens. Beyond that, we encounter run-on transcription artefacts or multi-sentence chunks that have lost their prosodic boundary. Training on very long segments also increases GPU memory pressure. If you have a specific podcast with notably long sentences, you could extend this to 150.
- **Why filter on German length only?** We do not impose a hard ceiling on the English side, because English translations of German compound sentences are often slightly longer. We do flag extreme length asymmetry (English > 2× German tokens) as a quality signal.

In [ ]:
# ── Filtering utilities ──────────────────────────────────────────────────────
MIN_DE_TOKENS = 5
MAX_DE_TOKENS = 120
MAX_LENGTH_RATIO = 2.5   # English / German token ratio upper bound

def token_length(text: str) -> int:
    """Return the number of tokens for a given text string."""
    return len(tokeniser.encode(text, add_special_tokens=False))

def is_valid_pair(de: str, en: str) -> bool:
    """Return True if the sentence pair passes all quality filters."""
    if not de or not en:
        return False
    de = de.strip()
    en = en.strip()
    if len(de) < 5 or len(en) < 5:
        return False
    de_len = token_length(de)
    en_len = token_length(en)
    if de_len < MIN_DE_TOKENS or de_len > MAX_DE_TOKENS:
        return False
    if en_len < MIN_DE_TOKENS:
        return False
    ratio = en_len / de_len
    if ratio > MAX_LENGTH_RATIO or ratio < (1 / MAX_LENGTH_RATIO):
        return False
    return True

def normalise_pair(de: str, en: str):
    """Strip whitespace and return a canonical (de, en) tuple."""
    return de.strip(), en.strip()

def make_fingerprint(de: str, en: str) -> str:
    """SHA-256 fingerprint for deduplication."""
    return hashlib.sha256(f'{de}|||{en}'.encode()).hexdigest()

print('Filter functions defined.')

---
## Download: CoVoST-2 (DE→EN)

CoVoST-2 is derived from Mozilla Common Voice. It contains transcriptions of crowdsourced voice recordings paired with human translations into English. The CC0 licence is the most permissive possible — you can do anything with it. The register is genuinely conversational: speakers read short prompts aloud, which produces natural sentence rhythms similar to podcast speech.

**How to cite:** Wang et al., "CoVoST 2 and Massively Multilingual Speech-to-Text Translation", 2021.

In [ ]:
# ── CoVoST-2 DE→EN ───────────────────────────────────────────────────────────
# The HuggingFace dataset ID is facebook/covost2; language pair is de_en.
# We load all splits and pool them — the splits here are based on the original
# Common Voice speaker splits, not our final train/val/test split.
print('Loading CoVoST-2 DE→EN ...')
covost_raw = load_dataset('facebook/covost2', 'de_en', trust_remote_code=True)

covost_pairs = []
for split in covost_raw.keys():
    for row in tqdm(covost_raw[split], desc=f'CoVoST-2 {split}'):
        de = row.get('sentence', '')
        en = row.get('translation', '')
        if is_valid_pair(de, en):
            de, en = normalise_pair(de, en)
            covost_pairs.append({'de': de, 'en': en, 'source': 'covost2'})

print(f'CoVoST-2: {len(covost_pairs):,} valid pairs after filtering')

---
## Download: MuST-C v3 (DE→EN)

MuST-C contains human translations of TED Talk transcripts. TED talks are the closest publicly available proxy for podcast speech: they are informal, use conversational grammar, include disfluencies (filled pauses are usually edited out in the transcripts), and cover diverse topics. The licence is CC BY-NC-ND 4.0, which means you cannot redistribute the data commercially, but you can train and distribute a *model* trained on it — the restriction applies to the data itself, not model weights.

**Important:** MuST-C requires accepting the data agreement at https://ict.fbk.eu/must-c/. The HuggingFace dataset `IWSLT/mustc` provides access to v3 programmatically if you have accepted the licence.

In [ ]:
# ── MuST-C v3 DE→EN ──────────────────────────────────────────────────────────
# If you have not accepted the MuST-C data agreement, this cell will fail with
# an access error. In that case, you can substitute with CoVoST-2 alone and
# adjust the blend proportions below.
print('Loading MuST-C v3 DE→EN ...')
try:
    mustc_raw = load_dataset(
        'IWSLT/mustc',
        'de-en',
        trust_remote_code=True
    )
    mustc_pairs = []
    for split in mustc_raw.keys():
        for row in tqdm(mustc_raw[split], desc=f'MuST-C {split}'):
            # MuST-C rows have 'src_text' and 'tgt_text' columns
            de = row.get('src_text', row.get('de', ''))
            en = row.get('tgt_text', row.get('en', ''))
            if is_valid_pair(de, en):
                de, en = normalise_pair(de, en)
                mustc_pairs.append({'de': de, 'en': en, 'source': 'mustc'})
    print(f'MuST-C v3: {len(mustc_pairs):,} valid pairs after filtering')
except Exception as e:
    print(f'MuST-C load failed: {e}')
    print('Falling back to CoVoST-2 only. Adjust BLEND_WEIGHTS below.')
    mustc_pairs = []

---
## Download: OPUS-Books (DE→EN)

OPUS-Books is a high-quality collection of translated literature from the OPUS corpus. It contains human-translated texts rather than machine-aligned subtitles, and its German is standard and varied. For our purposes it fills in vocabulary that MuST-C and CoVoST-2 under-represent. Note: the primary dataset loaded here is `Helsinki-NLP/opus_books`, with OPUS-100 as a fallback.

We apply strict filters: length range, length ratio, and a minimum unique character count to exclude mostly-punctuation segments. We also subsample to avoid it dominating the training blend.

In [ ]:
# ── OPUS-Books DE→EN ────────────────────────────────────────────────────────────
# We load a subset for speed; opus_books is a large dataset.
# If you want the full dataset, set streaming=True and iterate.
OPUS_BOOKS_SUBSAMPLE = 500_000   # Pairs to sample before filtering

print('Loading OPUS-Books DE→EN (this may take a few minutes)...')
try:
    opus_books_raw = load_dataset(
        'Helsinki-NLP/opus_books',
        'de-en',
        trust_remote_code=True,
        split='train'
    )
    opus_books_pairs = []
    count = 0
    for row in tqdm(opus_books_raw, desc='OPUS-Books'):
        if count >= OPUS_BOOKS_SUBSAMPLE:
            break
        try:
            de = row['translation']['de']
            en = row['translation']['en']
        except (KeyError, TypeError):
            continue
        count += 1
        if is_valid_pair(de, en):
            de, en = normalise_pair(de, en)
            opus_books_pairs.append({'de': de, 'en': en, 'source': 'opus_books'})
    print(f'OPUS-Books: {len(opus_books_pairs):,} valid pairs after filtering')
except Exception as e:
    print(f'OPUS-Books load failed: {e}')
    print('Trying direct OPUS-100 as fallback...')
    try:
        opus100 = load_dataset('Helsinki-NLP/opus-100', 'de-en',
                               trust_remote_code=True, split='train')
        opus_books_pairs = []
        for row in tqdm(opus100, desc='OPUS-100'):
            de = row['translation']['de']
            en = row['translation']['en']
            if is_valid_pair(de, en):
                de, en = normalise_pair(de, en)
                opus_books_pairs.append({'de': de, 'en': en, 'source': 'opus100'})
        print(f'OPUS-100: {len(opus_books_pairs):,} valid pairs after filtering')
    except Exception as e2:
        print(f'OPUS-100 also failed: {e2}')
        opus_books_pairs = []

In [ ]:
# ── Blend datasets ───────────────────────────────────────────────────────────
# Target proportions:
#   MuST-C     40%  — best register match, human verified
#   CoVoST-2   30%  — CC0, diverse speakers
#   OPUS-Books 30%  — scale, varied vocabulary
#
# If MuST-C is unavailable, redistribute proportionally between CoVoST-2 and
# OPUS-Books (55% / 45%).

TARGET_TOTAL = 300_000   # Adjust if you have more Colab quota / Drive space

all_sources = {
    'mustc': mustc_pairs,
    'covost2': covost_pairs,
    'opus_books': opus_books_pairs,
}

if mustc_pairs:
    target_counts = {
        'mustc': int(TARGET_TOTAL * 0.40),
        'covost2': int(TARGET_TOTAL * 0.30),
        'opus_books': int(TARGET_TOTAL * 0.30),
    }
else:
    target_counts = {
        'mustc': 0,
        'covost2': int(TARGET_TOTAL * 0.55),
        'opus_books': int(TARGET_TOTAL * 0.45),
    }

blended = []
for source_name, pairs in all_sources.items():
    target = target_counts[source_name]
    if not pairs or target == 0:
        continue
    sample = random.sample(pairs, min(len(pairs), target))
    blended.extend(sample)
    print(f'{source_name}: {len(sample):,} pairs sampled (available: {len(pairs):,})')

random.shuffle(blended)
print(f'\nTotal blended pairs: {len(blended):,}')

In [ ]:
# ── Deduplication ────────────────────────────────────────────────────────────
# We deduplicate on (German, English) pair fingerprints. This catches exact
# duplicates that appear in multiple OPUS snapshots. We do NOT do fuzzy
# deduplication here for speed reasons, but in a production pipeline you
# would also remove near-duplicates with MinHash or SimHash.
seen_fingerprints = set()
deduped = []
for pair in tqdm(blended, desc='Deduplication'):
    fp = make_fingerprint(pair['de'], pair['en'])
    if fp not in seen_fingerprints:
        seen_fingerprints.add(fp)
        deduped.append(pair)

print(f'After deduplication: {len(deduped):,} pairs ({len(blended) - len(deduped):,} removed)')

---
## Decision note: Train/validation/test split

We use an 88% / 9% / 3% split, which deviates slightly from the conventional 80/10/10 split. Here is why:

- **Test set is fixed at 500 pairs** from spoken-language sources (MuST-C + CoVoST-2). We want a meaningful number of test examples without wasting data. 500 pairs is sufficient for statistically significant COMET and chrF++ estimates.
- **Validation set is 3,000 pairs** — large enough to give a stable loss estimate every N steps and to compute chrF++ reliably, but small enough that evaluation during training does not become a bottleneck.
- The remainder goes to training. Given that we are fine-tuning a 1.5B model with LoRA (not training from scratch), more training data is better up to a point — but beyond ~100K high-quality pairs we expect diminishing returns.

In [ ]:
# ── Train / validation / test split ──────────────────────────────────────────
TEST_SIZE = 500       # Sacred held-out set — never touched during training
VAL_SIZE  = 3_000     # Validation set for monitoring training progress

# Prioritise spoken-language sources for the sacred test set
spoken_pairs = [p for p in deduped if p['source'] in ('mustc', 'covost2')]
other_pairs  = [p for p in deduped if p['source'] not in ('mustc', 'covost2')]

random.shuffle(spoken_pairs)
random.shuffle(other_pairs)

if len(spoken_pairs) >= TEST_SIZE:
    test_set  = spoken_pairs[:TEST_SIZE]
    remaining = spoken_pairs[TEST_SIZE:] + other_pairs
else:
    # Fall back to taking test from anywhere
    test_set  = deduped[:TEST_SIZE]
    remaining = deduped[TEST_SIZE:]

random.shuffle(remaining)
val_set   = remaining[:VAL_SIZE]
train_set = remaining[VAL_SIZE:]

print(f'Train: {len(train_set):,}')
print(f'Val:   {len(val_set):,}')
print(f'Test:  {len(test_set):,}  ← SACRED: never used during training')
print(f'\nTest set source distribution: {Counter(p["source"] for p in test_set)}')

In [ ]:
# ── Save to Google Drive ─────────────────────────────────────────────────────
import json

def save_jsonl(data: list, path: str):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'Saved {len(data):,} items → {path}')

save_jsonl(train_set, os.path.join(DATA_DIR, 'train.jsonl'))
save_jsonl(val_set,   os.path.join(DATA_DIR, 'val.jsonl'))
save_jsonl(test_set,  os.path.join(DATA_DIR, 'test.jsonl'))

# Save metadata
metadata = {
    'tokeniser': TOKENIZER_NAME,
    'filter': {'min_de_tokens': MIN_DE_TOKENS, 'max_de_tokens': MAX_DE_TOKENS,
               'max_length_ratio': MAX_LENGTH_RATIO},
    'split_sizes': {'train': len(train_set), 'val': len(val_set), 'test': len(test_set)},
    'sources': dict(Counter(p['source'] for p in deduped)),
}
with open(os.path.join(DATA_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)
print('Metadata saved.')

---
## Exploratory Data Analysis

Before training, it is worth spending a few minutes looking at the data. The cells below show length distributions, vocabulary statistics, and random examples. If anything looks wrong here, it is far better to catch it now than after a 10-hour training run.

In [ ]:
# ── Length distributions ─────────────────────────────────────────────────────
sample_size = min(5000, len(train_set))
sample = random.sample(train_set, sample_size)

de_lengths = [token_length(p['de']) for p in tqdm(sample, desc='Tokenising DE')]
en_lengths = [token_length(p['en']) for p in tqdm(sample, desc='Tokenising EN')]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(de_lengths, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('German segment length (tokens)')
axes[0].set_xlabel('Token count')
axes[0].set_ylabel('Frequency')

axes[1].hist(en_lengths, bins=40, color='coral', edgecolor='white')
axes[1].set_title('English translation length (tokens)')
axes[1].set_xlabel('Token count')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'length_distribution.png'), dpi=150)
plt.show()

print(f'DE — mean: {sum(de_lengths)/len(de_lengths):.1f}, '
      f'median: {sorted(de_lengths)[len(de_lengths)//2]}, '
      f'p95: {sorted(de_lengths)[int(len(de_lengths)*0.95)]}')
print(f'EN — mean: {sum(en_lengths)/len(en_lengths):.1f}, '
      f'median: {sorted(en_lengths)[len(en_lengths)//2]}, '
      f'p95: {sorted(en_lengths)[int(len(en_lengths)*0.95)]}')

In [ ]:
# ── Vocabulary statistics ─────────────────────────────────────────────────────
# Type-token ratio (TTR) is a rough proxy for lexical diversity.
# A low TTR suggests a repetitive corpus; a high TTR suggests diverse vocabulary.
# For reference: literary text typically has TTR ~0.4–0.6; conversational text ~0.3–0.5.
de_words = []
en_words = []
for p in sample:
    de_words.extend(p['de'].lower().split())
    en_words.extend(p['en'].lower().split())

de_ttr = len(set(de_words)) / len(de_words)
en_ttr = len(set(en_words)) / len(en_words)

print(f'German  — unique words: {len(set(de_words)):,}, TTR: {de_ttr:.3f}')
print(f'English — unique words: {len(set(en_words)):,}, TTR: {en_ttr:.3f}')

# Top 20 most common German words (should be function words, not proper nouns)
de_counter = Counter(de_words)
print('\nTop 20 German words:')
print(de_counter.most_common(20))

In [ ]:
# ── Random examples ───────────────────────────────────────────────────────────
# Inspect 10 random examples from each split. Read these carefully.
# Red flags to look for:
#   - German and English appear swapped
#   - Garbled characters or encoding artefacts
#   - Machine-generated text that slipped through
#   - Segment breaks in the middle of a sentence
print('=== 10 RANDOM TRAINING EXAMPLES ===\n')
for i, pair in enumerate(random.sample(train_set, 10), 1):
    print(f'[{i}] Source: {pair["source"]}')
    print(f'    DE: {pair["de"]}')
    print(f'    EN: {pair["en"]}')
    print()

print('=== 5 RANDOM TEST EXAMPLES (SACRED) ===\n')
for i, pair in enumerate(random.sample(test_set, 5), 1):
    print(f'[{i}] Source: {pair["source"]}')
    print(f'    DE: {pair["de"]}')
    print(f'    EN: {pair["en"]}')
    print()

In [ ]:
# ── Source distribution in final splits ──────────────────────────────────────
for split_name, split_data in [('train', train_set), ('val', val_set), ('test', test_set)]:
    dist = Counter(p['source'] for p in split_data)
    total = len(split_data)
    print(f'{split_name} ({total:,} pairs):')
    for src, count in sorted(dist.items()):
        print(f'  {src}: {count:,} ({count/total*100:.1f}%)')
    print()

---
## Summary and Next Steps

At this point you should have three `.jsonl` files in your Google Drive under `podcast_translation/data/`:

- `train.jsonl` — the training corpus
- `val.jsonl`   — used for validation during SFT and GRPO
- `test.jsonl`  — the sacred test set; do not load this until `04_evaluation.ipynb`

Check the length distribution plot and a handful of examples before proceeding. If anything looks off, adjust the filter parameters at the top of the notebook and re-run.

**Proceed to: `02_sft_training.ipynb`**